# LAB3 — Bài 6: Giảm số màu ảnh bằng k-Means

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Ảnh RGB mẫu được tạo ngay trong notebook để chạy trên Colab.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu 6. Tạo và hiển thị ảnh mẫu

Tạo một ảnh RGB nhỏ trực tiếp trong notebook để quy trình chạy được trên Colab, in kích thước và hiển thị ảnh.

In [2]:
in_danh_tinh()
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
# Tạo ảnh RGB mẫu ngay trong notebook để không phụ thuộc tệp cục bộ.
height,width=160,240
y,x=np.mgrid[0:height,0:width]
original=np.zeros((height,width,3),dtype=np.uint8)
original[:,:,0]=(45+130*x/width).astype(np.uint8)
original[:,:,1]=(90+95*y/height).astype(np.uint8)
original[:,:,2]=(190-110*y/height).astype(np.uint8)
original[(x-52)**2+(y-48)**2<27**2]=[255,205,45]
original[y>102+13*np.sin(x/18)]=[45,120,60]
original[y>127+9*np.sin(x/14+1)]=[28,82,43]
original[(x-180)**2+(y-105)**2<28**2]=[190,65,50]
OUT_DIR=Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
IMAGE_PATH=OUT_DIR/'sample_image_original.png'
Image.fromarray(original).save(IMAGE_PATH)
print('Kích thước (cao, rộng, kênh):',original.shape)
plt.figure(figsize=(8,5)); plt.imshow(original); plt.title('Ảnh gốc'); plt.axis('off'); plt.show()

Trần Chí Nguyên - 6451071052


Kích thước (cao, rộng, kênh): (160, 240, 3)


  plt.figure(figsize=(8,5)); plt.imshow(original); plt.title('Ảnh gốc'); plt.axis('off'); plt.show()


## Câu 6. Chuẩn hóa, gom cụm và khôi phục ảnh

Đưa màu về [0,1], reshape thành (cao × rộng, 3), dùng k-Means với 32 cụm và ánh xạ mỗi pixel về tâm cụm gần nhất.

In [3]:
in_danh_tinh()
X=original.astype('float32')/255.0
X_flat=X.reshape(-1,3)
N_COLORS=32
km=KMeans(n_clusters=N_COLORS,random_state=42,n_init=10)
labels=km.fit_predict(X_flat)
centers=np.clip(km.cluster_centers_,0,1)
quantized=centers[labels].reshape(height,width,3)
print('Số màu ban đầu:',len(np.unique(original.reshape(-1,3),axis=0)))
print('Số màu sau gom cụm:',len(np.unique((quantized*255).round().astype('uint8').reshape(-1,3),axis=0)))
fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].imshow(original); axes[0].set_title('Ảnh gốc'); axes[1].imshow(quantized); axes[1].set_title(f'k-Means, tối đa {N_COLORS} màu')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


Số màu ban đầu: 10265
Số màu sau gom cụm: 32


  plt.tight_layout(); plt.show()


## Câu 6. Lưu ảnh giảm màu và so sánh kích thước tệp

Lưu kết quả PNG dạng palette để ghi tối đa 32 màu vào tệp; so sánh kích thước với ảnh gốc.

In [4]:
in_danh_tinh()
output_path=OUT_DIR/'sample_image_kmeans_32.png'
indices=labels.reshape(height,width).astype('uint8')
palette=np.zeros((256,3),dtype='uint8')
palette[:N_COLORS]=(centers*255).round().astype('uint8')
palette_image=Image.fromarray(indices,mode='P')
palette_image.putpalette(palette.reshape(-1).tolist())
palette_image.save(output_path,optimize=True)
print('Ảnh gốc:',IMAGE_PATH, f'{IMAGE_PATH.stat().st_size:,} bytes')
print('Ảnh giảm màu:',output_path, f'{output_path.stat().st_size:,} bytes')
print('Tỷ lệ kích thước:',f'{output_path.stat().st_size/IMAGE_PATH.stat().st_size:.2%}')
plt.figure(figsize=(8,5)); plt.imshow(Image.open(output_path).convert('RGB')); plt.title('Ảnh đã lưu — 32 màu'); plt.axis('off'); plt.show()

Trần Chí Nguyên - 6451071052
Ảnh gốc: outputs\sample_image_original.png 2,355 bytes
Ảnh giảm màu: outputs\sample_image_kmeans_32.png 2,023 bytes
Tỷ lệ kích thước: 85.90%


  plt.figure(figsize=(8,5)); plt.imshow(Image.open(output_path).convert('RGB')); plt.title('Ảnh đã lưu — 32 màu'); plt.axis('off'); plt.show()
